# <center>Big Data for Engineers &ndash; Exercise 7 &ndash; Solutions</center>
## <center>Spring 2026 &ndash; Week 7 &ndash; ETH Zurich</center>
## <center>Massive Parallel Processing - MapReduce</center>

## 1. MapReduce theory
### 1.1 Theory questions
1. When reading a PB-sized dataset for a map task, is it usually kept as a huge HDFS file or is it sharded? Why? Are we forced to adopt one strategy over the other?
2. Given a partitioned dataset, how can we describe the dataflow of a query executed on it? What are the extreme cases? What is the impact on the execution?
3. When is it possible to use a reduce function as accumulate function without changing the results? Give some examples.
4. What are the fundamental arguments to be passed in every map and reduce functions? What is the relationship between the data types?
5. During a MapReduce query execution, when can reduce functions be called? Why? What about accumulate functions?

### Solutions:
1. The common practice in MapReduce is to split the input dataset into GB-sized files (**shards**). This allows handling the dataset more easily,
   for instance in case of network faults during downloads, while keeping the block-based intra-file parallelism, since a file is still **bigger** than an HDFS block.
    Additionally, partitioning the output enables each reduce worker to write to its own file independently without the need of concatenating the whole output.
   Theoretically speaking, HDFS would allow to store PB-sized files, so both strategies are possible.
2. A query executed on a partitioned dataset can be represented as a tree of operators, with every node representing an operation on the input data,
   and the edges representing the data movement: in some edges, there is a 1:1 mapping between
   each input partition and the corresponding output partition. In some other parts, output partitions depend on more than one input partition.
       
   <img src="https://polybox.ethz.ch/index.php/s/DAfpYztSw9AsqRT/download" width="30%">

   An extreme situation is when the whole computation is a **1:1** dependency between partitions:

    <img src="https://polybox.ethz.ch/index.php/s/JkWinqosqr9Qc2N/download" width="30%">

   while the opposite is where 
   each output partition depends on the **entire** input dataset.
       
   <img src="https://polybox.ethz.ch/index.php/s/B57tw7A6mkdDWtR/download" width="30%">


   During execution, a 1:1 dependency enables computing an output partition as soon as the corresponding input partition
   is available, without the need to wait for the other partitions to finish. Otherwise, the output can be computed only when all the input partitions are available.
3. Using a reduce function as combine function is possible whenever the reduce operation is **commutative** and **associative** and the intermediate key-value pairs
   have the same type as the output key-value pairs. This is because there is no guarantee on
   the input data and the order of the calls to the combine function (it could even *never* be called).
   Examples of commutative and associative reductions are the sum, the product, the minimum, and the maximum of the values.
   Conversely, the unweighted average cannot be implemented with a reduce function.
4. In general, a map function requires a key and a value as arguments, while a reduce function takes a key and a list of values as input.
    The key and value types for the reduce function must **coincide** with the types emitted by the map function.
5. In the MapReduce execution model, reduce functions can be called only **after** the map phase has completed entirely, because each reduce call needs all the values associated with
   an intermediate key. On the other hand, combine functions are called during the map phase without necessarily waiting for the completion of all the map tasks.

### 1.2 Combine feasibility
Given the following map-reduce functions, which of these can have its reduce function also used as a combine function?
Justify your answers.

1. `key` is the constant 1, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sum(values))
```

2. `key` is the constant 1, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sort(values)[0])
```
3. `key` is a float, `value` is an integer
```javascript
function map(key, value):
    emit(1, value)

function reduce(key, values[]):
    emit(1, sum(values)/len(values))
```

4. `key` is a string, `value` is a string (the + operator is string concatenation)
```javascript
function map(key, value):
    emit(key, value)

function reduce(key, values[]):
    res = key
    for val in sort(values):
        res = res + val
    res = res + key
    emit(key, res)
```

### Solutions
1. **Yes**. The function computes the sum across all values which is clearly associative and commutative.
2. **Yes**. The function computes the minimum of all values.
3. **No**. The function computes the unweighted average, which is not associative.
4. **No**. The function concatenates the sorted strings contained in `values` with key appended on both sides. Concatenating a partial set of strings would 
    break the global ordering, and the sequences will be interleaved by key, instead of having it only at the boundaries.

### 1.3 Terminology
Give a definition of the following terms in the context of MapReduce
- function
- task
- slot
- phase

For all of these terms, specify if they can be defined for map, reduce and combine.

### Solutions
- **function**: mathematical or programmed function that computes a certain output given a certain input.
  Map functions take a key-value pair as input and output zero, one or more intermediate key-value pairs.
  Reduce functions take as input one or more intermediate key-value pairs and output zero, one or more output key-value pairs.
  Combine functions take as input one or more intermediate key-value pairs and output zero, one or more intermediate key-value pairs.
- **task**: assignment consisting of multiple sequential function calls. There are map tasks and reduce tasks but not combine tasks (combine being called during the map phase).
    There is one map task for each input partition, and one reduce task for each intermediate result partition.
    
    <img src="https://polybox.ethz.ch/index.php/s/7H4NW5BbEiE9p4K/download" width="30%">
    
- **slot**: pool of computational resources with a CPU core and a partition of device memory reserved to it. A slot executes several tasks sequentially. There are map slots and
  reduce slots but not combine slots. The number of slots is limited by the number of CPU cores.

    <img src="https://polybox.ethz.ch/index.php/s/bWWngGzpG7KEis5/download" width="50%">
  
- **phase**: grouping of slots of the same kind, each executing in parallel. There is a map phase and a reduce phase but not a combine phase; instead, we have a shuffling phase.
  The shuffling phase can begin before the map phase is completed, while the reduce phase needs to wait for the map phase to reach completion.

## 1.4 Execution simulation
In this exercise, we will simulate the execution of a MapReduce job. To start with, we introduce the initial settings:
- Two TaskTrackers with 2 CPU cores each.
- Resources partitioned as 3 map slots and 1 reduce slot.
- One million input key-value pairs, partitioned into five chunks of 100k pairs each and two chunks of 250k pairs each.
- Each call to a map function takes 0.1ms.
- Each call to a reduce function takes 0.1ms for each value.
- Moving an intermediate key-value pair during shuffling takes 0.1ms.

We want to calculate the total runtime.

### Map phase
The job tracker assigns the first tasks to the three slots in this way:
- map slot1: task1, 100k
- map slot2: task2, 250k
- map slot3: task3, 250k

Left: four tasks of 100k.
    
The remaining tasks are assigned to a slot as soon as it finishes the current task. If two slots finish at the same time, the task is assigned to the one with lower ID.
Your aim is to calculate the schedule of each map slot.
    
You can keep track of the schedule by extending this table:
| Slot  | T = 0ms |
|-------|---------|
| Slot1 | t1:100k |
| Slot2 | t2:250k |
| Slot3 | t3:250k |

### Solution
The first slot that terminates is slot1, at T=0.1ms * 100k = 10s, and starts executing t4:100k. The other two tasks have processed 100k pairs each, so they are left with 150k pairs.
| Slot  | T = 0ms | T = 10s  |
|-------|---------|----------|
| Slot1 | t1:100k | t4:100k  |
| Slot2 | t2:250k | t2:150k  |
| Slot3 | t3:250k | t3:150k  |

Again, Slot1 terminates t4 after 10s and starts t5, the other two tasks have 50k pairs left to compute.
| Slot  | T = 0ms | T = 10s | T = 20s |
|-------|---------|---------|---------|
| Slot1 | t1:100k | t4:100k | t5:100k |
| Slot2 | t2:250k | t2:150k | t2:50k  |
| Slot3 | t3:250k | t3:150k | t3:50k  |

Proceeding this way, we come to this table:
| Slot  | T = 0ms | T = 10s | T = 20s | T = 25s | T = 30s | T = 35s |
|-------|---------|---------|---------|---------|---------|---------|
| Slot1 | t1:100k | t4:100k | t5:100k | t5:50k  | -       | -       |
| Slot2 | t2:250k | t2:150k | t2:50k  | t6:100k | t6:50k  | -       |
| Slot3 | t3:250k | t3:150k | t3:50k  | t7:100k | t7:50k  | -       |

Notice that Slot1 remains idle because there are no more map tasks left.

### Shuffling phase
The map tasks have output two key-value pairs for each key-value pair of input (one every 0.05ms).
Remember that shuffling can proceed in parallel with the map phase. Assume that as soon as a map function emits a pair, this can be shuffled.
Assume that inside a slot the intermediate pairs are moved sequentially.
    
When does the shuffling phase terminate? You can round the runtime to the millisecond.
Does it matter if a map slot resides on the same node with a reduce slot?

### Solution
Because shuffling can proceed in parallel with the map phase, it begins as soon as the slots write a pair. Each pair takes 0.1ms, so
the data is read more slowly than the speed at which the map functions output new data. We can simply look at the
slots that finish last (Slot2 and Slot3) and compute the time it takes to move all the pairs they output.
    
Slot2 and Slot3 each take (250k + 100k) = 350k pairs as input, so each outputs 350k*2 = 700k pairs. Shuffling takes 700k * 0.1ms = 70s, beginning at 0.05ms.

Notice that in this model we do not benefit from short-circuiting the data movement on the same machine: the reduce slot resides with only one map slot,
so at least Slot2 or Slot3 will reside on the other node, and it will require this amount of time to move the data over the network.

### Reduce phase
Once the shuffling phase has terminated, the reduce phase can begin. There are two million key value pairs to process.

What is the total runtime of the MapReduce job?

### Solution
The reduce function takes 0.1ms to process every value, and there is only one reduce slots.
This means that all the calls to the reduce function will be sequential.

The reduce phase runtime is simply 0.1ms * 2e6 = 200s.

The MapReduce job runtime is 70s + 200s = 270s.

### 1.5 Physical architecture
MapReduce V1 has two main process types: **JobTracker** and **TaskTracker**. Insert them in the table below using the parallelism with HBase and HDFS
| Master     | Worker       |
|------------|--------------|
| NameNode   | DataNode     |
| HMaster    | RegionServer |
| JobTracker | TaskTracker  |

<img src="https://polybox.ethz.ch/index.php/s/rndrSd9GfNfJSwX/download" width="30%">

As in HBase, we want the TaskTracker process to be running on DataNodes for **short-circuiting**:
if the input data partition is already stored on the DataNode that is running the TaskTracker,
we can process it without any network transfer.
However, the **shuffling phase** always requires all the TaskTrackers to connect to each other in order to exchange the
intermediate data.

## 2 MapReduce practice
### 2.1 Abstract functions
In this session, we will implement different map-reduce jobs using Python with a syntax that closely matches the pseudocode.
You can define a map task as a function that takes a key, a value and a context object as inputs and emits output pairs using
the `emit` method of the context object.
To test your solutions, you can execute the task using `submit(data, mapper, reducer)` where `data` is a list of key-value
pairs, `mapper` is the map function and `reducer` is the reduce function.

In [ ]:
from collections import defaultdict

class Context:
    def __init__(self):
        self.pairs = []

    def merge(self):
        merged = defaultdict(list)
        for key, value in self.pairs:
            merged[key].append(value)
        return merged.items()

    def emit(self, key, value):
        self.pairs.append((key, value))
        

def submit(pairs, mapper, reducer):
    map_context = Context()
    reduce_context = Context()
    for key, value in pairs:
        mapper(key, value, map_context)
    
    for key, values in map_context.merge():
        reducer(key, values, reduce_context)

    return reduce_context.pairs

In [ ]:
# Example usage: count the occurrencies of each key
def mapper(key, value, ctx):
    ctx.emit(key, None)

def reducer(key, values, ctx):
    ctx.emit(key, len(values))

data = [('a', None), ('a', None), ('b', None)]
submit(data, mapper, reducer)

[('a', 2), ('b', 1)]

In [ ]:
# 1. Return the unique keys of the input key-value pairs as [(key1, 1), ..., (keyN, 1)]

def mapper(key, value, ctx):
    ctx.emit(key, 1)

def reducer(key, values, ctx):
    ctx.emit(key, 1)

data = [('a', None), ('a', None), ('b', None)]
submit(data, mapper, reducer)

[('a', 1), ('b', 1)]

In [ ]:
# 2. Group by key and compute max, min, avg of the values as (key, (max, min, avg))

def mapper(key, value, ctx):
    ctx.emit(key, value)

def reducer(key, values, ctx):
    ctx.emit(key, (max(values), min(values), sum(values)/len(values)))

data = [('a', 1), ('b', 2), ('a', 2)]
submit(data, mapper, reducer)

[('a', (2, 1, 1.5)), ('b', (2, 2, 2.0))]

In [ ]:
# 3. count the number of words in a document with length above 5. A document might be chunked into several pairs
#    You can consider the space " " as the only word delimiter.
#    Input: [(doc_id, text),...]
#    Output: [(doc_id, n_words),]

def mapper(key, value, ctx):
    words = value.split(" ")
    filtered = [w for w in words if len(w) > 5]
    ctx.emit(key, len(filtered))

def reducer(key, values, ctx):
    ctx.emit(key, sum(values))

data = [(1, "Lorem ipsum"), (1, "dolorem"), (2, "a b c d")]
submit(data, mapper, reducer)

[(1, 1), (2, 0)]

In [ ]:
# 4. Count the occurencies of letters (case insensitive) in a document (0 if not occurring), sorted alphabetically (a document might come in chunks)
#    Hint: the builtin function ord(string) returns the ASCII number of string if string is a single character
#          and chr(integet) returns the corresponding char.
#    Input: [(doc_id, text),...]
#    Output: [(doc_id, [n_a, ..., n_z]),]

BASE = ord("a")
END = ord("z")
ALPHABET = [chr(i) for i in range(BASE, END+1)]

def mapper(key, value, ctx):
    chars = [t for t in value.lower() if t.isalpha()]
    count = []
    for char in ALPHABET:
        count.append(chars.count(char))
    ctx.emit(key, count)

def reducer(key, values, ctx):
    counter = [0]*26
    for partial in values:
        for i, c in enumerate(partial):
            counter[i] += c
    ctx.emit(key, counter)
    
data = [(1, "aabbb"), (1, "acc."), (2, "zz z")]
print(submit(data, mapper, reducer))

[(1, [3, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), (2, [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3])]


In [ ]:
# 5. Compute the scalar product of two vectors (hint: you could need more than one map-reduce job)
#    Input: [(position, value), ...]. The input sequence contains both vectors, encoded as pairs (i, v_i).
#    Output: [(1, A * B)]

def map1(key, value, ctx):
    ctx.emit(key, value)

def reduce1(key, values, ctx):
    a, b = values
    ctx.emit(key, a*b)

# We define a second map to keep submit simple, but potentially we could just 
# have just reduce2 and output (1, a*b) straight from reduce1

def map2(key, value, ctx):
    ctx.emit(1, value)

def reduce2(key, values, ctx):
    ctx.emit(1, sum(values))

data = [(1, 1), (2, 3), (1, -1), (2, 4)]
interm = submit(data, map1, reduce1)
submit(interm, map2, reduce2)

[(1, 11)]

### 2.2 Reverse engineering
Imagine we have the following table `students`:
| student_id | first_name  | last_name | grade |
|------------|------------|-----------|-------|
| AAAA       | Ada        | Lovelace  | 8     |
| AAAB       | Alan       | Turing    | 7     |
| AAAC       | Haskell    | Curry     | 9     |
...

Imagine it is stored as JSON Line and it is fed into MapReduce as a sequence like `(student_id, {'first_name':name, 'last_name':surname, 'grade':grade})`.

Given the pseudocode of map and reduce functions, infer the equivalent SQL statement

1.
```javascript
function map(key, value):
    if value["first_name"] == "Ada":
        emit(1, key)

function reduce(key, values[]):
    emit(1, values)
```

This job is selecting the IDs of all students called "Ada" (with repetitions).

**SQL**
```sql
SELECT student_id
FROM students
WHERE first_name = 'Ada'
```

2.
```javascript
function map(key, value):
    emit(value["grade"], 1)

function reduce(key, values[]):
    emit(key, len(values))
```

For every grade, count the number of times it appears in the table.

**SQL**
```sql
SELECT grade, count(*)
FROM students
GROUP BY grade
```

3.
```javascript
function map(key, value):
    emit(key, value["grade"])

function reduce(key, values[]):
    c = sum(values)/len(values)
    if c < 5:
        emit(key, c)
```

Return the IDs of the students with an average below 5.

**SQL**
```sql
SELECT student_id, AVG(grade)
FROM students
GROUP BY student_id
HAVING AVG(grade) < 5
```

4.
```javascript
function map(key, value):
    emit(value["first_name"], 1)

function reduce(key, values[]):
    emit(key, 1))
```

Get the set of all names.


**SQL**
```sql
SELECT DISTINCT first_name
FROM students
```

5.
```javascript
function map(key, value):
    emit(key, value["grade"])

function reduce1(key, values[]):
    emit(1, sum(values)/len(values))

function reduce2(key, values[]):
    emit(1, max(values))
```

Call chain: `map -> reduce1 -> reduce2`.

Select the highest average grade across all students' averages.

**SQL**
```sql
SELECT MAX(mean)
FROM (
    SELECT AVG(grade) AS mean
    FROM students
    GROUP BY student_id
)
```